# Article Content Extraction and Content-Enriched Splits

This notebook follows the same idea as the clean split workflow: **do not change the split membership**. Instead, it enriches each existing final de-leaked split with a new `content` column extracted from `news_url`.

Outputs:

- `final_artifacts/article_content_cache.csv`
- `final_artifacts/article_content_extraction_summary.csv`
- `data_splits/train_valid_images_only_image_deleaked_with_content.csv`
- `data_splits/val_valid_images_only_image_deleaked_with_content.csv`
- `data_splits/test_valid_images_only_image_deleaked_with_content.csv`

Important: many old news URLs may be dead, blocked, redirected, paywalled, or changed. Missing content is expected and should be reported honestly.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import time
import hashlib
import warnings

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

DATA_SPLIT_DIR = Path('data_splits')
FINAL_ARTIFACT_DIR = Path('final_artifacts')
FINAL_ARTIFACT_DIR.mkdir(exist_ok=True)

CONTENT_CACHE_PATH = FINAL_ARTIFACT_DIR / 'article_content_cache.csv'

# Set to None for full extraction. Use a small number first if you want a quick smoke test.
MAX_ROWS_PER_SPLIT = None

# Be polite and avoid hammering websites.
REQUEST_TIMEOUT = 12
REQUEST_SLEEP_SECONDS = 0.25
MIN_CONTENT_CHARS = 250

USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/125.0 Safari/537.36'
)

print('Content cache:', CONTENT_CACHE_PATH)

Content cache: final_artifacts\article_content_cache.csv


d:\DM\DL\Deep-learning-Multimodal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_clean_split(split):
    candidates = [
        DATA_SPLIT_DIR / f'{split}_valid_images_only_image_deleaked.csv',
        DATA_SPLIT_DIR / f'{split}_valid_images_only.csv',
        DATA_SPLIT_DIR / f'{split}_multimodal.csv',
        DATA_SPLIT_DIR / f'{split}.csv',
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            df['split'] = split
            print(f'Loaded {split}: {path} rows={len(df)}')
            return df, path
    raise FileNotFoundError(f'No split file found for {split}. Checked: {candidates}')

train_content_base, train_content_path = load_clean_split('train')
val_content_base, val_content_path = load_clean_split('val')
test_content_base, test_content_path = load_clean_split('test')

split_sources = pd.DataFrame([
    {'split': 'train', 'file': str(train_content_path), 'rows': len(train_content_base)},
    {'split': 'val', 'file': str(val_content_path), 'rows': len(val_content_base)},
    {'split': 'test', 'file': str(test_content_path), 'rows': len(test_content_base)},
])
display(split_sources)

required = {'id', 'news_url', 'title', 'label'}
for split_name, df in [('train', train_content_base), ('val', val_content_base), ('test', test_content_base)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{split_name} split is missing required columns: {missing}')

Loaded train: data_splits\train_valid_images_only_image_deleaked.csv rows=4829
Loaded val: data_splits\val_valid_images_only_image_deleaked.csv rows=541
Loaded test: data_splits\test_valid_images_only_image_deleaked.csv rows=518


,split,file,rows
0,train,data_splits\train_valid_images_only_image_dele...,4829
1,val,data_splits\val_valid_images_only_image_deleak...,541
2,test,data_splits\test_valid_images_only_image_delea...,518


In [3]:
def normalize_url(url):
    if not isinstance(url, str) or not url.strip():
        return ''
    url = url.strip()
    if url.lower() in {'nan', 'none', 'null'}:
        return ''
    if not re.match(r'^https?://', url, flags=re.I):
        url = 'https://' + url
    return url

def cache_key(url):
    url = normalize_url(url)
    return hashlib.sha256(url.encode('utf-8')).hexdigest() if url else ''

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('\xa0', ' ')
    return text.strip()

def extract_body_from_html(html):
    soup = BeautifulSoup(html, 'lxml')
    for tag in soup(['script', 'style', 'noscript', 'svg', 'form', 'iframe', 'header', 'footer', 'nav', 'aside']):
        tag.decompose()

    # Prefer semantic article containers.
    candidates = []
    for selector in ['article', '[role="article"]', 'main']:
        candidates.extend(soup.select(selector))
    if not candidates:
        candidates = [soup]

    best_text = ''
    best_score = -1
    for node in candidates:
        paragraphs = []
        for p in node.find_all(['p', 'h1', 'h2'], recursive=True):
            txt = clean_text(p.get_text(' ', strip=True))
            if len(txt) < 30:
                continue
            lower = txt.lower()
            boilerplate = [
                'subscribe', 'sign up', 'cookie', 'privacy policy', 'advertisement',
                'all rights reserved', 'follow us', 'share this article', 'newsletter',
            ]
            if any(x in lower for x in boilerplate) and len(txt) < 120:
                continue
            paragraphs.append(txt)
        text = clean_text(' '.join(paragraphs))
        score = len(text)
        if score > best_score:
            best_text = text
            best_score = score
    return best_text

def fetch_article_content(url):
    normalized = normalize_url(url)
    if not normalized:
        return {
            'normalized_url': '',
            'content': '',
            'status': 'missing_url',
            'http_status': np.nan,
            'error': '',
        }
    try:
        response = requests.get(
            normalized,
            headers={'User-Agent': USER_AGENT, 'Accept-Language': 'en-US,en;q=0.9'},
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )
        http_status = response.status_code
        if http_status >= 400:
            return {
                'normalized_url': normalized,
                'content': '',
                'status': 'http_error',
                'http_status': http_status,
                'error': f'HTTP {http_status}',
            }
        content_type = response.headers.get('content-type', '')
        if 'html' not in content_type.lower() and response.text.count('<') < 10:
            return {
                'normalized_url': response.url,
                'content': '',
                'status': 'non_html',
                'http_status': http_status,
                'error': content_type,
            }
        content = extract_body_from_html(response.text)
        status = 'ok' if len(content) >= MIN_CONTENT_CHARS else 'too_short'
        return {
            'normalized_url': response.url,
            'content': content,
            'status': status,
            'http_status': http_status,
            'error': '',
        }
    except Exception as exc:
        return {
            'normalized_url': normalized,
            'content': '',
            'status': 'request_error',
            'http_status': np.nan,
            'error': repr(exc)[:300],
        }

In [4]:
def load_content_cache():
    if CONTENT_CACHE_PATH.exists():
        cache = pd.read_csv(CONTENT_CACHE_PATH)
        if 'url_key' in cache.columns:
            print(f'Loaded content cache rows={len(cache)}')
            return cache
    return pd.DataFrame(columns=[
        'url_key', 'normalized_url', 'content', 'status', 'http_status', 'error',
        'content_len_chars', 'fetched_at_utc'
    ])

def save_content_cache(cache):
    cache = cache.drop_duplicates('url_key', keep='last').reset_index(drop=True)
    cache.to_csv(CONTENT_CACHE_PATH, index=False)
    return cache

def enrich_split_with_content(df, split_name, cache):
    work = df.copy().reset_index(drop=True)
    if MAX_ROWS_PER_SPLIT is not None:
        work = work.head(int(MAX_ROWS_PER_SPLIT)).copy()

    existing = {str(row.url_key): row for row in cache.itertuples(index=False) if isinstance(row.url_key, str)}
    new_rows = []

    for row in tqdm(work.itertuples(index=False), total=len(work), desc=f'Content extraction {split_name}'):
        url = getattr(row, 'news_url', '')
        key = cache_key(url)
        if not key:
            continue
        if key in existing:
            continue
        result = fetch_article_content(url)
        new_rows.append({
            'url_key': key,
            'normalized_url': result['normalized_url'],
            'content': result['content'],
            'status': result['status'],
            'http_status': result['http_status'],
            'error': result['error'],
            'content_len_chars': len(result['content']),
            'fetched_at_utc': datetime.now(timezone.utc).isoformat(),
        })
        if len(new_rows) % 25 == 0:
            cache = save_content_cache(pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True))
            existing = {str(row.url_key): row for row in cache.itertuples(index=False) if isinstance(row.url_key, str)}
            new_rows = []
        time.sleep(REQUEST_SLEEP_SECONDS)

    if new_rows:
        cache = save_content_cache(pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True))

    join = cache[['url_key', 'content', 'status', 'http_status', 'error', 'content_len_chars', 'normalized_url']].copy()
    work['url_key'] = work['news_url'].map(cache_key)
    enriched = work.merge(join, on='url_key', how='left')
    enriched['content'] = enriched['content'].fillna('')
    enriched['content_available'] = enriched['content'].str.len().ge(MIN_CONTENT_CHARS)
    enriched['content_status'] = enriched['status'].fillna('not_attempted')
    enriched['content_len_words'] = enriched['content'].str.split().map(len)
    enriched['text_for_model'] = np.where(enriched['content_available'], enriched['content'], enriched['title'].fillna(''))
    enriched = enriched.drop(columns=['status'], errors='ignore')
    return enriched, cache

In [5]:
cache = load_content_cache()

train_with_content, cache = enrich_split_with_content(train_content_base, 'train', cache)
val_with_content, cache = enrich_split_with_content(val_content_base, 'val', cache)
test_with_content, cache = enrich_split_with_content(test_content_base, 'test', cache)

output_paths = {
    'train': DATA_SPLIT_DIR / 'train_valid_images_only_image_deleaked_with_content.csv',
    'val': DATA_SPLIT_DIR / 'val_valid_images_only_image_deleaked_with_content.csv',
    'test': DATA_SPLIT_DIR / 'test_valid_images_only_image_deleaked_with_content.csv',
}

train_with_content.to_csv(output_paths['train'], index=False)
val_with_content.to_csv(output_paths['val'], index=False)
test_with_content.to_csv(output_paths['test'], index=False)

print('Saved content-enriched splits:')
for split, path in output_paths.items():
    print(split, path)

Content extraction test: 100%|██████████| 518/518 [13:19<00:00,  1.54s/it]


Saved content-enriched splits:
train data_splits\train_valid_images_only_image_deleaked_with_content.csv
val data_splits\val_valid_images_only_image_deleaked_with_content.csv
test data_splits\test_valid_images_only_image_deleaked_with_content.csv


In [6]:
summary_rows = []
for split_name, df in [('train', train_with_content), ('val', val_with_content), ('test', test_with_content)]:
    summary_rows.append({
        'split': split_name,
        'rows': len(df),
        'content_available': int(df['content_available'].sum()),
        'content_coverage_%': round(100 * df['content_available'].mean(), 2),
        'median_content_chars': float(df.loc[df['content_available'], 'content_len_chars'].median()) if df['content_available'].any() else 0,
        'fallback_to_title': int((~df['content_available']).sum()),
    })

content_summary = pd.DataFrame(summary_rows)
content_summary.to_csv(FINAL_ARTIFACT_DIR / 'article_content_extraction_summary.csv', index=False)
display(content_summary)

status_summary = pd.concat([
    train_with_content.assign(split='train'),
    val_with_content.assign(split='val'),
    test_with_content.assign(split='test'),
]).groupby(['split', 'content_status']).size().reset_index(name='count')
status_summary.to_csv(FINAL_ARTIFACT_DIR / 'article_content_status_summary.csv', index=False)
display(status_summary)

example_cols = ['id', 'label', 'title', 'news_url', 'content_status', 'content_len_chars', 'content']
display(test_with_content.loc[test_with_content['content_available'], example_cols].head(5))

,split,rows,content_available,content_coverage_%,median_content_chars,fallback_to_title
0,train,4829,3886,80.47,2563.0,943
1,val,541,451,83.36,2385.0,90
2,test,518,429,82.82,2633.0,89


,split,content_status,count
0,test,http_error,3
1,test,ok,429
2,test,request_error,1
3,test,too_short,85
4,train,http_error,29
5,train,ok,3886
6,train,request_error,15
7,train,too_short,899
8,val,http_error,2
9,val,ok,451


,id,label,title,news_url,content_status,content_len_chars,content
0,gossipcop-921650,real,Megan Fox gets emotional on Hollywood Medium,https://www.dailymail.co.uk/tvshowbiz/article-...,ok,4151,'I wasn't anticipating this': Megan Fox gets e...
1,gossipcop-943749,real,2018 MTV Movie and TV Awards: Complete List of...,https://www.thewrap.com/2018-mtv-movie-tv-awar...,ok,5178,The 2018 MTV Movie & TV Awards aired on Monday...
2,gossipcop-893601,real,Tracee Ellis Ross To Host The 2017 American Mu...,https://deadline.com/2017/11/tracee-ellis-ross...,ok,1957,It will be a family affair for Tracee Ellis Ro...
3,gossipcop-904865,real,The Best Oprah Moments of 2018,https://www.oprahmag.com/entertainment/g256316...,ok,4006,The trick with holiday gifts for a host is und...
5,gossipcop-896210,real,Kylie Jenner Chops Off Hair Again Amid Pregnan...,https://popculture.com/reality-tv/2017/11/26/k...,ok,1505,"Kylie Jenner got a short haircut Saturday, opt..."


## How to use this content later

After this notebook has produced `_with_content.csv` files, future modeling cells can use:

- `content` when available,
- `text_for_model` as a safe fallback column that uses `content` if extracted, otherwise `title`.

This avoids changing the split and avoids dropping rows just because an old URL failed.